<a href="https://colab.research.google.com/github/JAENED13/JAENED13/blob/main/Criador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# 📦 1. CONFIGURAÇÃO INICIAL (Executar 1x)
# ============================================

!pip install -q diffusers transformers accelerate safetensors torch

import torch
import math
import os
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
from PIL import Image
from google.colab import files
import re

print("🔧 Carregando modelo SDXL (pode levar 2-3 minutos)...")

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)

print("✅ Modelo carregado com sucesso!\n")

# ============================================
# 📝 2. SISTEMA DE METADADOS PROFISSIONAL
# ============================================

def processar_metadados(prompt_base):
    """Gera título e keywords otimizados para sites de stock"""

    # Lista expandida de termos técnicos para remover
    termos_remover = [
        "8k", "4k", "photorealistic", "ultra", "detailed", "high res",
        "professional", "quality", "render", "photography", "shot",
        "hdr", "sharp", "crisp", "vivid", "stunning"
    ]

    # Criar título limpo
    titulo = prompt_base.lower()
    for termo in termos_remover:
        titulo = re.sub(rf'\b{termo}\b', '', titulo, flags=re.IGNORECASE)

    # Limpeza final do título
    titulo = re.sub(r'[,\-]+', ' ', titulo)  # Remove pontuações
    titulo = ' '.join(titulo.split())  # Remove espaços duplicados
    titulo = titulo.strip().title()  # Capitaliza cada palavra

    # Limitar a 100 caracteres (padrão stock sites)
    if len(titulo) > 100:
        titulo = titulo[:97] + "..."

    # Gerar keywords inteligentes
    palavras = re.findall(r'\b\w{4,}\b', prompt_base.lower())  # Palavras 4+ caracteres

    # Tags complementares por categoria
    tags_comerciais = [
        "stock photo", "commercial use", "high quality", "professional",
        "digital art", "concept", "design", "background", "isolated", "template"
    ]

    # Combinar e remover duplicatas mantendo ordem
    tags_unicas = []
    for palavra in palavras + tags_comerciais:
        if palavra not in tags_unicas and palavra not in termos_remover:
            tags_unicas.append(palavra)

    # Limitar a 10-15 tags (padrão stock)
    tags_finais = tags_unicas[:15]

    # Criar arquivo CSV para upload em massa
    conteudo_txt = f"""╔══════════════════════════════════════╗
║     METADADOS PARA SITE DE STOCK     ║
╚══════════════════════════════════════╝

📌 TÍTULO (100 caracteres máx):
{titulo}

🏷️ KEYWORDS ({len(tags_finais)} tags):
{', '.join(tags_finais)}

📄 DESCRIÇÃO SUGERIDA:
High-quality {titulo.lower()} suitable for commercial use. Perfect for websites, presentations, marketing materials, and creative projects.

📊 ESPECIFICAÇÕES TÉCNICAS:
- Resolução: 5MP (ideal para stock)
- Formato: PNG (alta qualidade)
- Licença: Royalty-free (definir conforme necessário)
- Categoria: [Definir manualmente]

═══════════════════════════════════════
Gerado automaticamente - Revisar antes do upload
"""

    # Criar também versão CSV para import em massa
    conteudo_csv = f"Filename,Title,Keywords,Category\nimagem_final_stock.png,\"{titulo}\",\"{', '.join(tags_finais)}\",General\n"

    # Salvar ambos os arquivos
    with open("metadados_imagem.txt", "w", encoding="utf-8") as f:
        f.write(conteudo_txt)

    with open("upload_batch.csv", "w", encoding="utf-8") as f:
        f.write(conteudo_csv)

    print("\n" + "="*50)
    print(conteudo_txt)
    print("="*50)

# ============================================
# 🎨 3. WORKFLOW COMPLETO DE GERAÇÃO
# ============================================

def workflow_estoque_completo():
    """Fluxo completo: Prompt → Imagem 5MP → Metadados → Download"""

    print("╔════════════════════════════════════════════╗")
    print("║  GERADOR DE IMAGENS PARA SITES DE STOCK   ║")
    print("╚════════════════════════════════════════════╝\n")

    # 1️⃣ Input do usuário
    prompt = input("📝 Digite o prompt (em inglês, seja específico):\n> ")

    print("\n📐 Escolha a proporção:")
    print("  [1] Quadrado (1:1) - 1024x1024 → ~2800x2800px final")
    print("  [2] Paisagem (3:2) - 1216x832 → ~3300x2250px final")
    print("  [3] Retrato (2:3) - 832x1216 → ~2250x3300px final")

    opcao = input("\nOpção [1/2/3]: ").strip()

    # Dimensões base (otimizadas para SDXL)
    dimensoes = {
        '1': (1024, 1024),
        '2': (1216, 832),
        '3': (832, 1216)
    }

    dims = dimensoes.get(opcao, (1024, 1024))

    # 2️⃣ Geração da imagem
    print(f"\n🎨 Gerando imagem {dims[0]}x{dims[1]}...")
    print("⏳ Tempo estimado: 15-30 segundos\n")

    negative_prompt = """
    logo, watermark, text, signature, username, artist name,
    blurry, low quality, distorted, deformed, ugly, messy,
    bad anatomy, poorly drawn, amateur, grainy, noisy
    """

    imagem_base = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt.strip(),
        width=dims[0],
        height=dims[1],
        num_inference_steps=35,  # Aumentado para melhor qualidade
        guidance_scale=7.5
    ).images[0]

    # 3️⃣ Upscale para 5MP
    print("🚀 Redimensionando para 5 Megapixels (padrão stock)...")

    target_pixels = 5_000_000
    aspect_ratio = dims[0] / dims[1]

    nova_altura = int(math.sqrt(target_pixels / aspect_ratio))
    nova_largura = int(nova_altura * aspect_ratio)

    # Upscale com Lanczos (melhor qualidade)
    imagem_final = imagem_base.resize(
        (nova_largura, nova_altura),
        Image.Resampling.LANCZOS
    )

    print(f"✅ Dimensões finais: {nova_largura}x{nova_altura}px ({nova_largura*nova_altura/1_000_000:.1f}MP)")

    # 4️⃣ Salvar com máxima qualidade
    nome_arquivo = "imagem_final_stock.png"
    imagem_final.save(
        nome_arquivo,
        format="PNG",
        optimize=False,  # PNG sem compressão
        compress_level=1  # Compressão mínima
    )

    # 5️⃣ Gerar metadados
    print("\n📋 Gerando metadados...")
    processar_metadados(prompt)

    # 6️⃣ Download automático
    print("\n📥 Iniciando downloads...\n")

    arquivos = [
        nome_arquivo,
        "metadados_imagem.txt",
        "upload_batch.csv"
    ]

    for arquivo in arquivos:
        if os.path.exists(arquivo):
            print(f"  ⬇️ {arquivo}")
            files.download(arquivo)

    print("\n✅ PROCESSO CONCLUÍDO!")
    print("\n💡 Próximos passos:")
    print("  1. Revise os metadados gerados")
    print("  2. Ajuste categoria/descrição se necessário")
    print("  3. Faça upload em Shutterstock/Adobe Stock/etc")

# ============================================
# ▶️ EXECUTAR GERADOR
# ============================================

workflow_estoque_completo()

🔧 Carregando modelo SDXL (pode levar 2-3 minutos)...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

AssertionError: Torch not compiled with CUDA enabled